# 05 — Isolation Forest (Phase 2)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name=='notebooks' else str(Path.cwd()))
from src.utils.data_loader import DatasetSource
from src.ml.baseline import load_series, add_event_labels
from src.ml.isolation_forest import run_iforest_experiment
import pandas as pd

src = DatasetSource()
ei = src.read_csv('Wind Farm A/event_info.csv', sep=';')
FEATURES = ['sensor_0_avg','wind_speed_3_avg','sensor_5_avg','power_30_avg','sensor_11_avg','sensor_13_avg']


In [ ]:
# Single experiment: anomaly event 0, contamination 0.05
df = load_series(src, 'A', 0, asset_id=0)
df = add_event_labels(df, ei, 0)
m, df = run_iforest_experiment(df, FEATURES, contamination=0.05)
print(m)


In [ ]:
# Contamination sweep on anomaly event 0
rows=[]
for c in [0.01,0.03,0.05,0.1,0.15]:
    d = load_series(src, 'A', 0, asset_id=0)
    d = add_event_labels(d, ei, 0)
    m,_ = run_iforest_experiment(d, FEATURES, contamination=c)
    rows.append(m)
print(pd.DataFrame(rows)[['contamination','precision','recall','f1','false_alarms_per_day']].to_string(index=False))


In [ ]:
# Cross-event evaluation, contamination 0.05
agg=[]
for _, ev in ei.iterrows():
    eid=int(ev['event_id']); asset=int(ev['asset'] if 'asset' in ev else ev['asset_id'])
    try:
        d = load_series(src, 'A', eid, asset_id=asset)
        d = add_event_labels(d, ei, eid)
        m,_ = run_iforest_experiment(d, FEATURES, contamination=0.05)
        m['event_id']=eid; m['label']=ev['event_label']
        agg.append(m)
    except Exception as e:
        agg.append({'event_id':eid,'label':ev['event_label'],'error':str(e)})
rdf = pd.DataFrame(agg)
print(rdf.groupby('label')[['precision','recall','f1','false_alarms_per_day']].mean().to_string())
rdf.to_csv('reports/iforest_results.csv', index=False)
print('saved reports/iforest_results.csv')
